# WRDS Extract 02: CRSP Monthly Panel for S&P 500 Members

Run `01_wrds_extract_membership.ipynb` first so that `sp500_membership_monthly.parquet` already exists.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import wrds
from pandas.tseries.offsets import MonthEnd

In [2]:
start_date = pd.Timestamp("2000-01-01")
end_date = pd.Timestamp("2025-12-31")
outdir = Path("data/raw")
crsp_table = "crsp.msf"
crsp_event_table = "crsp.mseall"
crsp_names_table = "crsp.msenames"
membership_file = outdir / "sp500_membership_monthly.parquet"

# Leave these as None if you want WRDS to use its default login flow.
wrds_username = None
wrds_password = None

if not membership_file.exists():
    raise FileNotFoundError(f"Missing membership parquet: {membership_file}")

start_date, end_date, membership_file

(Timestamp('2000-01-01 00:00:00'),
 Timestamp('2025-12-31 00:00:00'),
 PosixPath('data/raw/sp500_membership_monthly.parquet'))

In [3]:
if wrds_username is not None and wrds_password is not None:
    db = wrds.Connection(wrds_username=wrds_username, wrds_password=wrds_password)
elif wrds_username is not None:
    db = wrds.Connection(wrds_username=wrds_username)
else:
    db = wrds.Connection()

db

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
membership_monthly = pd.read_parquet(membership_file)
membership_monthly["month_end"] = pd.to_datetime(membership_monthly["month_end"])
permnos = sorted(membership_monthly["permno"].dropna().astype(int).unique().tolist())

if not permnos:
    raise RuntimeError("The membership parquet exists, but it contains no PERMNOs.")

len(permnos)

1062

In [5]:
crsp_sql = f"""
    SELECT
        msf.permno::integer AS permno,
        msf.\"date\"::date AS date,
        msf.ret,
        msf.retx,
        mse.dlret,
        msf.prc,
        msf.shrout,
        msf.vol,
        names.exchcd,
        names.shrcd
    FROM {crsp_table} AS msf
    LEFT JOIN {crsp_event_table} AS mse
      ON msf.permno = mse.permno
     AND msf.\"date\" = mse.\"date\"
    LEFT JOIN {crsp_names_table} AS names
      ON msf.permno = names.permno
     AND names.namedt <= msf.\"date\"
     AND msf.\"date\" <= COALESCE(names.nameendt, CAST('9999-12-31' AS date))
    WHERE msf.permno = ANY(CAST(%(permnos)s AS integer[]))
      AND msf.\"date\"::date BETWEEN CAST(%(start_date)s AS date) AND CAST(%(end_date)s AS date)
    ORDER BY msf.permno, msf.\"date\"
"""

crsp = db.raw_sql(
    crsp_sql,
    params={
        "permnos": permnos,
        "start_date": start_date.date(),
        "end_date": end_date.date(),
    },
)

if crsp.empty:
    raise RuntimeError(
        f"CRSP query returned zero rows from {crsp_table!r}. "
        "The table may be unavailable for this WRDS account."
    )

crsp.head()

,permno,date,ret,retx,dlret,prc,shrout,vol,exchcd,shrcd
0,10078,2000-01-31,0.014528,0.014528,<NA>,78.5625,1561106.0,4163042.0,3,11
1,10078,2000-02-29,0.21241,0.21241,<NA>,95.25,1561106.0,3131641.0,3,11
2,10078,2000-03-31,-0.01624,-0.01624,<NA>,93.70313,1586413.0,3521264.0,3,11
3,10078,2000-04-28,-0.018843,-0.018843,<NA>,91.9375,1588626.0,5067037.0,3,11
4,10078,2000-05-31,-0.166553,-0.166553,<NA>,76.625,1588626.0,4110327.0,3,11


In [6]:
crsp["permno"] = pd.to_numeric(crsp["permno"], errors="coerce").astype("Int64")
crsp["date"] = pd.to_datetime(crsp["date"], errors="coerce")
crsp = crsp.dropna(subset=["permno", "date"]).copy()

if crsp.empty:
    raise RuntimeError(
        "CRSP rows were returned, but none had valid permno/date values."
    )

crsp["permno"] = crsp["permno"].astype(int)
crsp["month_end"] = crsp["date"] + MonthEnd(0)

numeric_cols = ["ret", "retx", "dlret", "prc", "shrout", "vol", "exchcd", "shrcd"]
for col in numeric_cols:
    crsp[col] = pd.to_numeric(crsp[col], errors="coerce")

crsp["me"] = crsp["prc"].abs() * crsp["shrout"]
both_missing = crsp["ret"].isna() & crsp["dlret"].isna()
crsp["retadj"] = (1.0 + crsp["ret"].fillna(0.0)) * (
    1.0 + crsp["dlret"].fillna(0.0)
) - 1.0
crsp.loc[both_missing, "retadj"] = np.nan

member_keys = membership_monthly[["permno", "month_end"]].drop_duplicates()
crsp_panel = (
    crsp.merge(
        member_keys, on=["permno", "month_end"], how="inner", validate="many_to_one"
    )
    .sort_values(["month_end", "permno"])
    .reset_index(drop=True)
)

if crsp_panel.empty:
    raise RuntimeError("The CRSP/member-month join produced zero rows.")

crsp_panel.head()

,permno,date,ret,retx,dlret,prc,shrout,vol,exchcd,shrcd,month_end,me,retadj
0,10078,2000-01-31,0.014528,0.014528,<NA>,78.5625,1561106.0,4163042.0,3,11,2000-01-31,122644390.125,0.014528
1,10104,2000-01-31,-0.108477,-0.108477,<NA>,49.95313,2847344.0,5257006.0,3,11,2000-01-31,142233744.98672,-0.108477
2,10107,2000-01-31,-0.16167,-0.16167,<NA>,97.875,5160025.0,6483703.0,3,11,2000-01-31,505037446.875,-0.16167
3,10138,2000-01-31,0.052454,0.052454,<NA>,38.875,120678.0,88960.0,3,11,2000-01-31,4691357.25,0.052454
4,10145,2000-01-31,-0.167931,-0.167931,<NA>,48.0,789233.0,771383.0,1,11,2000-01-31,37883184.0,-0.167931


In [7]:
crsp_outfile = outdir / "crsp_monthly_sp500_panel.parquet"
crsp_panel.to_parquet(crsp_outfile, engine="pyarrow", index=False)

print(f"saved: {crsp_outfile}")
print(f"rows: {len(crsp_panel):,}")
print(f"permnos: {crsp_panel['permno'].nunique():,}")

saved: data/raw/crsp_monthly_sp500_panel.parquet
rows: 150,900
permnos: 1,062


In [8]:
db.close()